In [8]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")


embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

embed_result = embeddings.embed_query("Hello AI")
print(len(embed_result))
print(embed_result[:10])

c:\Users\Echo\Desktop\KNA_AgenticAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


384
[-0.03338824212551117, 0.03453981503844261, 0.05947450175881386, 0.059286098927259445, -0.06353531032800674, -0.06819586455821991, 0.08823321014642715, 0.0344407856464386, -0.03278516232967377, -0.01581495814025402]


In [10]:
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()
os.environ['GOOGLE_API_KEY']=os.getenv("GOOGLE_API_KEY")

embeddings = GoogleGenerativeAIEmbeddings(model='models/embedding-001')

embed_result = embeddings.embed_query("Hello AI")
print(len(embed_result))
print(embed_result[:10])

768
[0.01908188685774803, -0.04838583618402481, -0.0033970330841839314, -0.015432494692504406, 0.026915451511740685, -0.037899527698755264, 0.03683774173259735, 0.0024586953222751617, 0.031139591708779335, -0.002974053379148245]


In [14]:
from pinecone import Pinecone
import os

os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')

pc = Pinecone()


In [ ]:
from pinecone import ServerlessSpec

## Serverless: Server will be managed by the vendor
index_name = "kna-agenticai"

## Creating a Index
if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = 768,
        metric = "cosine",
        spec = ServerlessSpec(cloud = "aws", region = "us-east-1")
    )

In [19]:
## Loading the Index
index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore

## Creating Vector Store
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [23]:
from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [ ]:
## Universal Identification number
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['bcc748e9-3085-486a-b422-df068d6c382b',
 '06cbc1b8-17cc-41a3-90a7-9b1ba98a217e',
 '90fce4c6-758c-47de-9e8b-7a37907af925',
 '0dddba0e-8207-4d49-a517-0d61c08d58f7',
 'a2ea5191-c6ff-4267-96d6-f68dbc7f6ca2',
 '463605ab-4021-4681-bc16-005876bee758',
 'c4843c4f-70b9-4da6-9777-6085dd27fec8',
 'bbeba6a1-0441-4255-9929-43eae2e61358',
 '7c07c25c-172e-436c-975f-b93fdb1c1083',
 '655e74e1-7cab-45a5-9b24-4d5143e06773']

In [31]:
results = vector_store.similarity_search("What langchain provides to us?", k=2)
results

[Document(id='90fce4c6-758c-47de-9e8b-7a37907af925', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='bbeba6a1-0441-4255-9929-43eae2e61358', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [33]:
results = vector_store.similarity_search("What langchain provides to us?", filter= {'source': 'news'})
results

[Document(id='7c07c25c-172e-436c-975f-b93fdb1c1083', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='0dddba0e-8207-4d49-a517-0d61c08d58f7', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(id='06cbc1b8-17cc-41a3-90a7-9b1ba98a217e', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.')]

In [44]:
retriever=vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 3, "score_threshold": 0.5}
)
retriever.invoke("google")

[Document(id='90fce4c6-758c-47de-9e8b-7a37907af925', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='655e74e1-7cab-45a5-9b24-4d5143e06773', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='bbeba6a1-0441-4255-9929-43eae2e61358', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [45]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain import hub
import pprint

model=ChatGoogleGenerativeAI(model='gemini-1.5-flash')

prompt = hub.pull("rlm/rag-prompt")
pprint.pprint(prompt.messages)

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


In [52]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

rag_chain.invoke("Use of Langchain?")

"Langchain is used for building projects,  specifically mentioned is an exciting new project.  The provided text also suggests it's useful for creating stateful, agentic applications."

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [51]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(template = "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:",
                          input_variables = ['context', 'question']
                          )

prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:")